# Build Dataset v3

Build `kaya-go/moku-v3` by combining:
1. **v2 real data** — Existing corrected real images from `kaya-go/moku-v2` (config=real)
2. **Generated images** — Synthetic-conditioned Gemini images with inherited annotations, verified via annotator (`data/annotate_generated/`)

**No synthetic data** — v3 drops the pure synthetic split entirely. The synthetic
images are only used as conditioning inputs for Gemini; only the photorealistic
outputs enter the dataset.

**Strategy**: Keep v2 test/val sets identical so metrics are directly comparable.
New generated images go into **train only**.

**Categories:**
| ID | Name | Description |
|-----|------|------|
| 0 | black_stone | Individual black stone |
| 1 | white_stone | Individual white stone |
| 2 | board_corner | Board corner point (small bbox at each corner) |

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from datasets import DatasetDict, concatenate_datasets, load_dataset
from IPython.display import display

from moku.dataset import (
    ID_TO_CATEGORY,
    compute_split_stats,
    load_annotated_generated,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Step 1: Load v2 Real Dataset from HF Hub

Test and validation splits are kept as-is for comparable metrics.

In [3]:
HF_DATASET_V2 = "kaya-go/moku-v2"

v2_real = load_dataset(HF_DATASET_V2, name="real")

print("v2 real:", v2_real)

v2 real: DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 382
    })
    validation: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 53
    })
    test: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 50
    })
})


## Step 2: Load Generated Images (Verified)

Load the synthetic-conditioned Gemini images with verified annotations.

In [5]:
ANNOTATE_DIR = Path("../data/annotate_generated")
IMAGES_DIR = ANNOTATE_DIR / "images"
IMAGES_JSON = ANNOTATE_DIR / "images.json"

generated_ds = load_annotated_generated(
    images_dir=IMAGES_DIR,
    images_json_path=IMAGES_JSON,
)

print(f"Generated images loaded: {len(generated_ds)}")
print(generated_ds)

Generated images loaded: 1000
Dataset({
    features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
    num_rows: 1000
})


## Step 3: Build v3 Dataset

- **train**: v2 real train + all generated images
- **validation**: v2 real validation (unchanged)
- **test**: v2 real test (unchanged)

No synthetic split — only real + generated photorealistic images.

In [6]:
v3_real = DatasetDict({
    "train": concatenate_datasets([v2_real["train"], generated_ds]),
    "validation": v2_real["validation"],
    "test": v2_real["test"],
})

print("v3 real:", v3_real)

v3 real: DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 1382
    })
    validation: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 53
    })
    test: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 50
    })
})


## Step 4: Dataset Statistics

Compare v2 vs v3 real datasets and review the synthetic data.

In [7]:
stats_rows = []
for label, ds_dict in [("v2-real", v2_real), ("v3", v3_real)]:
    for split_name, split_ds in ds_dict.items():
        stats = compute_split_stats(split_ds)
        stats_rows.append({
            "dataset": label,
            "split": split_name,
            "images": stats["num_images"],
            "total_objects": stats["total_objects"],
            "avg_objects_per_image": round(stats["avg_objects_per_image"], 1),
        })

display(pd.DataFrame(stats_rows))

,dataset,split,images,total_objects,avg_objects_per_image
0,v2-real,train,382,25787,67.5
1,v2-real,validation,53,2378,44.9
2,v2-real,test,50,3179,63.6
3,v3,train,1382,172386,124.7
4,v3,validation,53,2378,44.9
5,v3,test,50,3179,63.6


In [8]:
# Per-category breakdown for v3
cat_rows = []
for split_name, split_ds in v3_real.items():
    stats = compute_split_stats(split_ds)
    for cat_id, count in stats["category_counts"].most_common():
        cat_rows.append({
            "split": split_name,
            "category": ID_TO_CATEGORY[cat_id],
            "count": count,
        })

display(pd.DataFrame(cat_rows))

,split,category,count
0,train,black_stone,83990
1,train,white_stone,82875
2,train,board_corner,5521
3,validation,black_stone,1205
4,validation,white_stone,962
5,validation,board_corner,211
6,test,black_stone,1602
7,test,white_stone,1377
8,test,board_corner,200


In [9]:
# Source dataset breakdown for v3 train
stats = compute_split_stats(v3_real["train"])
source_df = pd.DataFrame(
    [{"source": src, "count": cnt} for src, cnt in stats["source_counts"].most_common()]
)
display(source_df)

,source,count
0,generated,1000
1,go_chess,195
2,go_game_v10,187


## Browse Dataset

In [10]:
from moku.viz import browse_dataset

browse_dataset(v3_real)

## Step 5: Push to Hugging Face Hub

Push as `kaya-go/moku-v3` — single config, no synthetic split.

In [11]:
HF_DATASET_V3 = "kaya-go/moku-v3"

v3_real.push_to_hub(HF_DATASET_V3, private=False)
print(f"Dataset pushed to {HF_DATASET_V3}")
print(f"Available at: https://huggingface.co/datasets/{HF_DATASET_V3}")

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Map:   0%|          | 0/691 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Dataset pushed to kaya-go/moku-v3
Available at: https://huggingface.co/datasets/kaya-go/moku-v3
